In [ ]:
# Configuracion de cluster Slurm (alineada con run.sh / run_parallel.sh)
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_EXPERIMENT = 'shard:4'
SLURM_GRES_MASSIVE = 'shard:6'
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

print('Partition:', SLURM_PARTITION)
print('CPUs:', SLURM_CPUS_PER_TASK, '| Mem:', SLURM_MEM)
print('GRES exp:', SLURM_GRES_EXPERIMENT, '| GRES masivo:', SLURM_GRES_MASSIVE)
print('Conda env:', CONDA_ENV)
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ['PYTORCH_CUDA_ALLOC_CONF'])


# 05 - Experimento Audio VAD

Objetivo tecnico:
1. Extraer audio desde videos.
2. Eliminar silencios con VAD (WebRTCVAD o fallback con energia de Librosa).
3. Extraer embeddings acusticos con Wav2Vec2.
4. Entrenar regresion logistica en tres configuraciones de idioma (ES, EN, ES+EN).
5. Exportar tres JSON de prediccion sobre el conjunto completo disponible.


In [ ]:
def resolve_train_json(project_root: Path) -> Path:
    local_candidate = project_root.parent / 'materials' / 'dataset_task3_exist2026' / 'training.json'
    cluster_candidate = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training/training.json')
    if local_candidate.exists():
        return local_candidate
    if cluster_candidate.exists():
        return cluster_candidate
    raise FileNotFoundError('Training JSON not found in local materials or cluster path.')

PROJECT_ROOT = resolve_project_root()
TRAIN_JSON_PATH = resolve_train_json(PROJECT_ROOT)
TEST_JSON_PATH = PROJECT_ROOT.parent / 'materials' / 'dataset_task3_exist2026' / 'test.json'
if not TEST_JSON_PATH.exists():
    raise FileNotFoundError(f'test.json not found: {TEST_JSON_PATH}')

VIDEO_ROOT = TRAIN_JSON_PATH.parent
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
PREDICCIONES_FINALES_DIR = PROJECT_ROOT / 'predicciones_finales'
AUDIO_DIR = ARTIFACTS_DIR / 'audio_wav'
CACHE_NPZ = ARTIFACTS_DIR / 'audio_vad_w2v_features.npz'

for d in [ARTIFACTS_DIR, PREDICCIONES_FINALES_DIR, AUDIO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRAIN_JSON_PATH:', TRAIN_JSON_PATH)
print('TEST_JSON_PATH:', TEST_JSON_PATH)
print('HAS_WEBRTCVAD:', HAS_WEBRTCVAD)
        


In [ ]:
with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    raw_train = json.load(f)
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    raw_test = json.load(f)

rows = []
for sid, obj in raw_train.items():
    rec = {'sample_id': str(sid), 'source': 'train'}
    rec.update(obj)
    rows.append(rec)
for sid, obj in raw_test.items():
    rec = {'sample_id': str(sid), 'source': 'test'}
    rec.update(obj)
    rows.append(rec)

df = pd.DataFrame(rows)

def majority_task3_1(lbls):
    vals = [x for x in lbls if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    if c['YES'] == c['NO']:
        return vals[0]
    return c.most_common(1)[0][0]

df['label'] = df['labels_task3_1'].apply(majority_task3_1) if 'labels_task3_1' in df.columns else 'UNKNOWN'
df['y'] = df['label'].map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)
df['video_path_abs'] = df['path_video'].apply(lambda p: str((VIDEO_ROOT / p).resolve()))
df['lang'] = df['lang'].fillna('').astype(str).str.lower()

df = df.sort_values('sample_id').reset_index(drop=True)
print(df[['sample_id', 'lang', 'source']].head(2))
print('n=', len(df), '| langs=', df['lang'].value_counts().to_dict())
        


In [ ]:
# Decision de diseno:
# - WebRTCVAD produce un recorte mas estricto para voz hablada.
# - Si no esta disponible, se usa un fallback energetico puro NumPy (evita dependencia de librosa/numba).
def extract_audio_ffmpeg(video_path: str, wav_path: str, sr: int = 16000):
    if Path(wav_path).exists():
        return wav_path
    cmd = [
        'ffmpeg', '-y', '-i', video_path,
        '-vn', '-ac', '1', '-ar', str(sr), wav_path
    ]
    subprocess.run(cmd, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return wav_path


def read_mono_16k(wav_path: str):
    y, sr = sf.read(wav_path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    if sr != 16000:
        g = np.gcd(sr, 16000)
        y = resample_poly(y, up=16000 // g, down=sr // g).astype(np.float32)
        sr = 16000
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))
    return y, sr


def vad_webrtc(y: np.ndarray, sr: int, mode: int = 2):
    vad = webrtcvad.Vad(mode)
    if sr != 16000:
        raise ValueError('WebRTCVAD requires 16kHz audio.')

    y16 = np.clip(y, -1.0, 1.0)
    pcm = (y16 * 32767).astype(np.int16).tobytes()

    frame_ms = 30
    frame_len = int(sr * frame_ms / 1000) * 2
    voiced = []
    for i in range(0, len(pcm) - frame_len + 1, frame_len):
        frame = pcm[i:i + frame_len]
        if vad.is_speech(frame, sample_rate=sr):
            voiced.append(frame)

    if not voiced:
        return y

    out = np.frombuffer(b''.join(voiced), dtype=np.int16).astype(np.float32) / 32767.0
    return out


def vad_energy_numpy(y: np.ndarray, sr: int, frame_ms: int = 30, hop_ms: int = 15, q: float = 0.25):
    frame_len = max(int(sr * frame_ms / 1000), 1)
    hop_len = max(int(sr * hop_ms / 1000), 1)
    if len(y) < frame_len:
        return y

    energies = []
    starts = []
    for s in range(0, len(y) - frame_len + 1, hop_len):
        frame = y[s:s + frame_len]
        energies.append(float(np.mean(frame * frame)))
        starts.append(s)

    energies = np.array(energies, dtype=np.float32)
    thr = float(np.quantile(energies, q))
    keep = energies >= thr
    if not np.any(keep):
        return y

    chunks = [y[s:s + frame_len] for s, k in zip(starts, keep) if k]
    return np.concatenate(chunks).astype(np.float32)


def clean_audio_with_vad(wav_path: str):
    y, sr = read_mono_16k(wav_path)
    if HAS_WEBRTCVAD:
        try:
            yc = vad_webrtc(y, sr)
        except Exception:
            yc = vad_energy_numpy(y, sr)
    else:
        yc = vad_energy_numpy(y, sr)

    if yc is None or len(yc) < 3200:
        yc = y
    return yc.astype(np.float32), 16000

In [ ]:
# Embeddings acusticos Wav2Vec2 con promedio temporal del ultimo hidden state.
MODEL_NAME = 'facebook/wav2vec2-base-960h'
processor = AutoProcessor.from_pretrained(MODEL_NAME)
audio_model = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(DEVICE)
audio_model.eval()

if CACHE_NPZ.exists():
    cache = np.load(CACHE_NPZ, allow_pickle=True)
    X_audio = cache['X_audio']
    sid_cache = cache['sample_ids'].astype(str)
    assert list(sid_cache) == df['sample_id'].astype(str).tolist()
    print('Loaded cache:', CACHE_NPZ, '| shape=', X_audio.shape)
else:
    embs = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc='Audio VAD + Wav2Vec2'):
        wav_path = AUDIO_DIR / f"{row.sample_id}.wav"
        extract_audio_ffmpeg(row.video_path_abs, str(wav_path))

        y_clean, sr = clean_audio_with_vad(str(wav_path))
        inputs = processor(y_clean, sampling_rate=sr, return_tensors='pt', padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            hid = audio_model(**inputs).last_hidden_state
            emb = hid.mean(dim=1).squeeze(0).cpu().numpy()
        embs.append(emb)

    X_audio = np.vstack(embs).astype(np.float32)
    np.savez_compressed(CACHE_NPZ, X_audio=X_audio, sample_ids=df['sample_id'].astype(str).values)
    print('Saved cache:', CACHE_NPZ, '| shape=', X_audio.shape)


In [ ]:
def train_and_export_three_lang_models(X, y, langs, sample_ids, source, exp_tag):
    configs = {
        'ES': ['es'],
        'EN': ['en'],
        'ES_EN': ['es', 'en'],
    }

    for cfg_name, cfg_langs in configs.items():
        m_train = np.isin(langs, cfg_langs) & (y >= 0) & (source == 'train')
        m_test = np.isin(langs, cfg_langs) & (source == 'test')

        Xtr, ytr = X[m_train], y[m_train]
        if len(np.unique(ytr)) < 2:
            raise RuntimeError(f'{cfg_name}: subset sin dos clases para entrenamiento')
        if not m_test.any():
            raise RuntimeError(f'{cfg_name}: no hay filas en test.json para este idioma')

        clf = Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
        ])

        binc = np.bincount(ytr)
        n_splits = int(max(2, min(5, np.min(binc))))
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        scores = cross_val_score(clf, Xtr, ytr, cv=cv, scoring='f1_macro', n_jobs=-1)
        print(f'{cfg_name} CV F1 macro mean: {float(scores.mean()):.4f}')

        clf.fit(Xtr, ytr)

        pred = clf.predict(X[m_test])
        pred_labels = np.where(pred == 1, 'YES', 'NO')

        ids_out = sample_ids[m_test]
        out = [
            {'test_case': 'EXIST2026', 'id': str(sid), 'value': str(lbl)}
            for sid, lbl in zip(ids_out, pred_labels)
        ]
        out_path = PREDICCIONES_FINALES_DIR / f'BeingChillingWeWillWin_{exp_tag}_{cfg_name}.json'
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(out, f, ensure_ascii=False)
        print('Saved', out_path, '| rows=', len(out))


train_and_export_three_lang_models(
    X_audio,
    df['y'].to_numpy(np.int64),
    df['lang'].astype(str).to_numpy(),
    df['sample_id'].astype(str).to_numpy(),
    df['source'].astype(str).to_numpy(),
    exp_tag='05AudioVAD_W2V2',
)
        
